# 05 — Benchmark: Base vs Prompt vs LoRA vs QLoRA

Este notebook compara las 4 rutas de calidad del Z2H-Shop Assistant:

| Ruta | Descripción |
|------|-------------|
| **Base** | Modelo base sin modificaciones |
| **Prompt Engineering** | Instrucciones mejoradas + ejemplos |
| **LoRA** | Fine-tuning ligero con adaptador |
| **QLoRA** | Fine-tuning con cuantización 4-bit |

### Métricas evaluadas
- **Calidad**: Score del evaluador LLM (1-5)
- **Costo entrenamiento**: Tiempo, GPU-horas, pérdida
- **Costo inferencia**: Latencia, tokens/s

## Setup

In [ ]:
import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

BASE_DIR = Path().resolve().parent
OLLAMA_ENDPOINT = os.getenv("OLLAMA_ENDPOINT", "http://localhost:11434")
MODEL_BASE = os.getenv("MODEL_BASE", "microsoft/phi-2")

print(f"Ollama: {OLLAMA_ENDPOINT}")
print(f"Modelo base: {MODEL_BASE}")

## 1. Preguntas de test

5 preguntas reales del dominio Z2H-Shop que cubren diferentes escenarios.

In [ ]:
TEST_QUERIES = [
    {
        "query": "Mi cliente recibió el producto dañado y quiere un reembolso urgente",
        "context": "Pedido #9876: funda de laptop, daño en esquina del empaque",
    },
    {
        "query": "¿Cómo puedo mejorar el ranking de mis productos en la categoría electrónica?",
        "context": "Vendedor con 3 meses de experiencia, 15 productos activos",
    },
    {
        "query": "El envío de mi último pedido se retrasó 5 días y el cliente está furioso",
        "context": "Envío estándar, destino: Lima, tracking sin actualizar",
    },
    {
        "query": "¿Qué hago si un cliente deja una reseña falsa de 1 estrella?",
        "context": "Producto con 4.5 estrellas promedio, 200+ reseñas",
    },
    {
        "query": "Necesito configurar envíos internacionales a Europa",
        "context": "Vendedor nuevo en categoría moda, sin experiencia en exportación",
    },
]

print(f"{len(TEST_QUERIES)} preguntas de test definidas")

## 2. Funciones de evaluación

In [ ]:
import httpx

def call_ollama(prompt, model=None):
    """Llama a Ollama y retorna (respuesta, tiempo_segundos)."""
    model = model or MODEL_BASE
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.7, "num_predict": 256},
    }
    start = time.time()
    try:
        resp = httpx.post(f"{OLLAMA_ENDPOINT}/api/generate", json=payload, timeout=120.0)
        elapsed = time.time() - start
        resp.raise_for_status()
        return resp.json().get("response", ""), elapsed
    except Exception as e:
        elapsed = time.time() - start
        return f"[Error: {e}]", elapsed

def call_vllm(prompt, port=8080):
    """Llama a vLLM (OpenAI-compatible)."""
    payload = {"model": "default", "prompt": prompt, "max_tokens": 256, "temperature": 0.7}
    start = time.time()
    try:
        resp = httpx.post(f"http://localhost:{port}/v1/completions", json=payload, timeout=120.0)
        elapsed = time.time() - start
        resp.raise_for_status()
        choices = resp.json().get("choices", [])
        return choices[0].get("text", "") if choices else "", elapsed
    except Exception as e:
        elapsed = time.time() - start
        return f"[Error: {e}]", elapsed

EVAL_TEMPLATE = (
    "Eres un evaluador experto. Evalúa la calidad de esta respuesta.\n\n"
    "Pregunta: {query}\nRespuesta: {response}\n\n"
    "Criterios: 5=perfecta, 4=buena, 3=aceptable, 2=mala, 1=muy mala\n"
    "Responde SOLO con un número del 1 al 5."
)

def evaluate_response(query, response):
    """Evalúa usando el modelo como juez."""
    prompt = EVAL_TEMPLATE.format(query=query, response=response)
    score_text, _ = call_ollama(prompt)
    for char in score_text.strip():
        if char.isdigit() and 1 <= int(char) <= 5:
            return float(char)
    return 3.0

## 3. Evaluar las 4 rutas

In [ ]:
PROMPT_TEMPLATE = (
    "Eres el asistente de soporte del Z2H-Shop. Ayuda al seller con empatía, "
    "pasos concretos y alternativas.\n\nConsulta: {query}\nContexto: {context}\nRespuesta:"
)

def run_bench(path_name, call_fn, **kwargs):
    """Ejecuta benchmark para una ruta dada."""
    results = []
    total_latency = 0.0
    for q in TEST_QUERIES:
        if path_name == "prompt_engineering":
            prompt = PROMPT_TEMPLATE.format(**q)
        else:
            prompt = q["query"]

        response, latency = call_fn(prompt, **kwargs)
        total_latency += latency
        score = evaluate_response(q["query"], response)
        results.append({"query": q["query"], "response": response, "score": score, "latency": latency})
        print(f"    Score: {score}, Latency: {latency:.2f}s")

    avg_score = sum(r["score"] for r in results) / len(results)
    avg_latency = total_latency / len(results)
    return results, avg_score, avg_latency, total_latency

In [ ]:
# 1. Modelo base
print("[1/4] Evaluando modelo base...")
base_results, base_score, base_latency, base_total = run_bench("base", call_ollama)

# 2. Prompt engineering
print("\n[2/4] Evaluando prompt engineering...")
pe_results, pe_score, pe_latency, pe_total = run_bench("prompt_engineering", call_ollama)

# 3. LoRA (via vLLM)
print("\n[3/4] Evaluando LoRA (vLLM)...")
lora_results, lora_score, lora_latency, lora_total = run_bench("lora", call_vllm)

# 4. QLoRA (via vLLM)
print("\n[4/4] Evaluando QLoRA (vLLM)...")
qlora_results, qlora_score, qlora_latency, qlora_total = run_bench("qlora", call_vllm)

## 4. Tabla comparativa

In [ ]:
# Cargar métricas de entrenamiento si existen
lora_train = {}
qlora_train = {}

lora_path = BASE_DIR / "lora_metrics.json"
qlora_path = BASE_DIR / "qlora_metrics.json"

if lora_path.exists():
    with open(lora_path, encoding="utf-8") as f:
        lora_train = json.load(f)
if qlora_path.exists():
    with open(qlora_path, encoding="utf-8") as f:
        qlora_train = json.load(f)

# Construir tabla
rows = [
    {
        "Ruta": "base",
        "Calidad (avg)": base_score,
        "Latencia (s)": base_latency,
        "Loss entrenamiento": "N/A",
        "Tiempo entrenamiento (s)": "N/A",
    },
    {
        "Ruta": "prompt_engineering",
        "Calidad (avg)": pe_score,
        "Latencia (s)": pe_latency,
        "Loss entrenamiento": "N/A",
        "Tiempo entrenamiento (s)": "N/A",
    },
    {
        "Ruta": "lora",
        "Calidad (avg)": lora_score,
        "Latencia (s)": lora_latency,
        "Loss entrenamiento": lora_train.get("train_loss", "N/A"),
        "Tiempo entrenamiento (s)": lora_train.get("train_runtime_seconds", "N/A"),
    },
    {
        "Ruta": "qlora",
        "Calidad (avg)": qlora_score,
        "Latencia (s)": qlora_latency,
        "Loss entrenamiento": qlora_train.get("train_loss", "N/A"),
        "Tiempo entrenamiento (s)": qlora_train.get("train_runtime_seconds", "N/A"),
    },
]

df = pd.DataFrame(rows)
df = df.sort_values("Calidad (avg)", ascending=False, kind="stable")
df = df.reset_index(drop=True)

print("TABLA COMPARATIVA")
print("=" * 80)
df.to_string(index=False)

In [ ]:
df

## 5. Gráficos comparativos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
rutas = df["Ruta"].tolist()

# 1. Calidad
axes[0].barh(rutas, df["Calidad (avg)"], color=colors)
axes[0].set_xlabel("Score promedio (1-5)")
axes[0].set_title("Calidad de respuesta")
axes[0].set_xlim(0, 5.5)
for i, v in enumerate(df["Calidad (avg)"]):
    axes[0].text(v + 0.05, i, f"{v:.2f}", va="center", fontweight="bold")

# 2. Latencia
axes[1].barh(rutas, df["Latencia (s)"], color=colors)
axes[1].set_xlabel("Latencia promedio (s)")
axes[1].set_title("Velocidad de inferencia")
for i, v in enumerate(df["Latencia (s)"]):
    axes[1].text(v + 0.05, i, f"{v:.2f}s", va="center", fontweight="bold")

# 3. Calidad vs Latencia
axes[2].scatter(df["Latencia (s)"], df["Calidad (avg)"], c=colors, s=200, zorder=5)
for i, row in df.iterrows():
    axes[2].annotate(row["Ruta"], (row["Latencia (s)"], row["Calidad (avg)"]),
                     textcoords="offset points", xytext=(10, 5))
axes[2].set_xlabel("Latencia (s)")
axes[2].set_ylabel("Calidad (avg)")
axes[2].set_title("Calidad vs Latencia")
axes[2].grid(True, alpha=0.3)

plt.suptitle("Benchmark — Z2H-Shop Assistant", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Análisis de costo de entrenamiento

In [ ]:
print("COSTO DE ENTRENAMIENTO")
print("=" * 50)

if lora_train:
    print(f"\nLoRA:")
    print(f"  Pérdida final:    {lora_train.get('train_loss', 'N/A')}")
    print(f"  Tiempo total:     {lora_train.get('train_runtime_seconds', 'N/A'):.1f}s")
    print(f"  Épocas:           {lora_train.get('epochs', 'N/A')}")
    print(f"  Dataset size:     {lora_train.get('dataset_size', 'N/A')}")
else:
    print("\nLoRA: Métricas no disponibles (ejecuta train_lora.py primero)")

if qlora_train:
    print(f"\nQLoRA:")
    print(f"  Pérdida final:    {qlora_train.get('train_loss', 'N/A')}")
    print(f"  Tiempo total:     {qlora_train.get('train_runtime_seconds', 'N/A'):.1f}s")
    print(f"  Épocas:           {qlora_train.get('epochs', 'N/A')}")
    print(f"  Cuantización:     {qlora_train.get('quantization', 'N/A')}")
    print(f"  Dataset size:     {qlora_train.get('dataset_size', 'N/A')}")
else:
    print("\nQLoRA: Métricas no disponibles (ejecuta train_qlora.py primero)")

if lora_train and qlora_train:
    lora_time = lora_train.get("train_runtime_seconds", 0)
    qlora_time = qlora_train.get("train_runtime_seconds", 0)
    if lora_time > 0 and qlora_time > 0:
        print(f"\nAhorro de tiempo QLoRA vs LoRA: {(1 - qlora_time/lora_time)*100:.1f}%")

## 7. Detalle por pregunta

In [ ]:
# Comparar respuestas por pregunta
all_results = {
    "base": base_results,
    "prompt": pe_results,
    "lora": lora_results,
    "qlora": qlora_results,
}

for i, q in enumerate(TEST_QUERIES):
    print(f"\n{'='*70}")
    print(f"Pregunta {i+1}: {q['query']}")
    print(f"{'='*70}")
    for path, results in all_results.items():
        r = results[i]
        print(f"\n[{path.upper()}] (score: {r['score']})")
        print(f"  {r['response'][:150]}...")

## 8. Decisión documentada

### Cuándo usar cada ruta

| Escenario | Ruta recomendada | Justificación |
|-----------|-----------------|---------------|
| Prototipo rápido | Prompt engineering | Sin costo de entrenamiento |
| Dominio específico | LoRA | Captura tono y formato del negocio |
| GPU limitada | QLoRA | Mismo resultado, menos VRAM |
| Datos cambiantes | LoRA/QLoRA | Reentrenar adaptador es barato |
| Calidad genérica suficiente | Base | Sin overhead |

### Regla de decisión

```
¿El modelo base responde correctamente sin contexto específico?
  → SÍ: usa modelo base
  → NO: ¿Tienes datos de dominio?
    → SÍ: ¿Tienes GPU?
      → SÍ: LoRA
      → NO: QLoRA (o CPU con tiempo)
    → NO: Prompt engineering
```

In [ ]:
# Guardar resultados
output = {
    "comparison": df.to_dict(orient="records"),
    "details": {
        "base": [{"query": r["query"], "score": r["score"]} for r in base_results],
        "prompt": [{"query": r["query"], "score": r["score"]} for r in pe_results],
        "lora": [{"query": r["query"], "score": r["score"]} for r in lora_results],
        "qlora": [{"query": r["query"], "score": r["score"]} for r in qlora_results],
    },
    "train_metrics": {
        "lora": lora_train,
        "qlora": qlora_train,
    },
}

json_path = BASE_DIR / "bench_results.json"
csv_path = BASE_DIR / "bench_results.csv"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

df.to_csv(csv_path, index=False, encoding="utf-8")

print(f"Resultados guardados:")
print(f"  JSON: {json_path}")
print(f"  CSV:  {csv_path}")
print("\n✅ Benchmark completado")

## Resumen

### Qué hicimos
1. Definimos 5 preguntas de test del dominio Z2H-Shop
2. Evaluamos 4 rutas: base, prompt engineering, LoRA, QLoRA
3. Medimos calidad (evaluador LLM), latencia y costo de entrenamiento
4. Generamos tablas comparativas y gráficos
5. Documentamos la decisión: cuándo fine-tunear vs prompt engineering

### Key takeaways
- Prompt engineering es la ruta de entrada más rápida y barata
- LoRA/QLoRA justifican el costo cuando hay datos de dominio específicos
- QLoRA ofrece la misma calidad que LoRA con menos VRAM
- La decisión final depende del trade-off calidad vs costo